In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
C   0.000   1.397   0.000
C   1.210  -0.698   0.000
C  -1.210  -0.698   0.000
N   0.000  -1.439   0.000
N   1.246   0.719   0.000
N  -1.246   0.719   0.000
H   0.000   2.426   0.000
H   2.101  -1.213   0.000
H  -2.101  -1.213   0.000'''
molecule = '1_3_5_C3H3N3'
spin = 0
charge =  0


In [6]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [4]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [7]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

overwrite output file: ./1_3_5_C3H3N3.txt
orbital  energy   occ
0     -15.578197007968717    2.0
1     -15.57818755055276    2.0
2     -15.578096969440004    2.0
3     -11.290678280830777    2.0
4     -11.290571872772949    2.0
5     -11.290477971287425    2.0
6     -1.25266958361085    2.0
7     -1.1220583809614004    2.0
8     -1.1215926357119836    2.0
9     -0.8611150526059793    2.0
10     -0.8607297774906622    2.0
11     -0.7269817938024542    2.0
12     -0.6244622292938067    2.0
13     -0.5687010122301815    2.0
14     -0.5685582392306352    2.0
15     -0.5147723640008336    2.0
16     -0.5121111743848039    2.0
17     -0.3545566694421788    2.0
18     -0.3543851308825232    2.0
19     -0.34803323891349625    2.0
20     -0.3475027921507634    2.0
21     0.1730058102271847    0.0
22     0.17357719322940923    0.0
23     0.3754437409324515    0.0
24     0.5463166043172023    0.0
25     0.5470048509561034    0.0
26     0.5890665999758791    0.0
27     0.6680295620737423    0.0
28

In [9]:
n_elec= 6
n_orb = 6 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([17,20,21,22,23,24])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-277.9322523581481),
 np.float64(-6.9262368737187785),
 FCIvector([[ 8.71057592e-01, -1.78789013e-02,  0.00000000e+00,
             -6.58552103e-05,  0.00000000e+00, -1.78257509e-02,
              0.00000000e+00,  9.35606315e-02,  0.00000000e+00,
              4.23535668e-05,  1.12449485e-04,  0.00000000e+00,
             -3.64767853e-03,  0.00000000e+00,  2.74839365e-02,
              0.00000000e+00, -8.16227036e-06,  0.00000000e+00,
              2.73229318e-02, -2.02239349e-03],
            [-1.78789013e-02, -2.07926107e-01,  0.00000000e+00,
             -2.00697207e-03,  0.00000000e+00, -4.79347013e-02,
              0.00000000e+00,  5.51767571e-03,  0.00000000e+00,
             -1.22299883e-02, -6.52516812e-03,  0.00000000e+00,
             -6.91813700e-02,  0.00000000e+00,  6.00674118e-04,
              0.00000000e+00,  1.25399835e-02,  0.00000000e+00,
              7.90281112e-04, -1.10531374e-02],
            [ 0.00000000e+00,  0.00000000e+00, -1.77274196e-02,
     

In [10]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      0.871057591697319
[0 1 2]      [0 1 3]      -0.017878901336287
[0 1 2]      [1 2 3]      -0.000065855210335
[0 1 2]      [0 2 4]      -0.017825750941980
[0 1 2]      [0 3 4]      0.093560631480645
[0 1 2]      [2 3 4]      0.000042353566821
[0 1 2]      [0 1 5]      0.000112449485112
[0 1 2]      [1 2 5]      -0.003647678526824
[0 1 2]      [1 3 5]      0.027483936541796
[0 1 2]      [0 4 5]      -0.000008162270357
[0 1 2]      [2 4 5]      0.027322931762834
[0 1 2]      [3 4 5]      -0.002022393493954
[0 1 3]      [0 1 2]      -0.017878901336286
[0 1 3]      [0 1 3]      -0.207926107140947
[0 1 3]      [1 2 3]      -0.002006972066910
[0 1 3]      [0 2 4]      -0.047934701307586
[0 1 3]      [0 3 4]      0.005517675713792
[0 1 3]      [2 3 4]      -0.012229988254870
[0 1 3]      [0 1 5]      -0.006525168118043
[0 1 3]      [1 2 5]      -0.069181370032974
[0 1 3]      [1 3 5]      0.000600674117550
[0 1 3]      [0 4 5]      0.012539983511676
[0 1 3]      [2 4 5